# MCP (Model Context Protocol)

本筆記本示範如何用 **FastMCP** 將工具封裝成標準化的 MCP Server，以及如何透過 LangGraph 的 MCP Client 串接並使用這些工具。

### 為什麼需要 MCP？

| 問題 | MCP 解法 |
|------|----------|
| 每個框架有自己的 Tool 格式 | 統一協議，一次實作處處可用 |
| 工具散落各處難以共用 | 封裝成獨立 Server，多 Agent 共用 |
| 新增工具需改 Agent 程式碼 | Server 端更新，Client 自動發現 |

> **類比**：MCP 就像 USB 介面，出現之前每種設備各自接口不相容；有了 MCP，所有 AI 工具都能即插即用。

In [ ]:
%%capture
!pip install fastmcp langchain-mcp-adapters langchain langchain-openai langgraph

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

---
## 1. 建立 MCP Server（hp_mcp_server.py）

我們把 HP 的三個業務工具封裝成一個 MCP Server：
- `check_inventory` — 查詢庫存
- `check_warranty` — 確認保固狀態
- `get_repair_sop` — 取得維修 SOP

In [ ]:
# 將 MCP Server 寫入檔案（稍後用 subprocess 執行）
server_code = '''
from fastmcp import FastMCP

mcp = FastMCP("HP Tools")

# 模擬資料庫
_INVENTORY = {
    "HP EliteBook 840": {"stock": 15, "location": "台北倉"},
    "HP LaserJet Pro": {"stock": 3,  "location": "新竹倉"},
    "HP ZBook Studio":  {"stock": 0,  "location": "缺貨"},
}
_WARRANTY = {
    "SN12345": {"status": "有效", "expire": "2026-12-31", "model": "HP EliteBook 840"},
    "SN99999": {"status": "過期", "expire": "2024-01-01", "model": "HP LaserJet Pro"},
}
_SOP = {
    "無法開機": "1. 確認電源線 2. 嘗試硬重啟 3. 進 BIOS 診斷 4. 聯絡技術支援",
    "印表機卡紙": "1. 關閉電源 2. 打開前蓋 3. 輕輕取出紙張 4. 重新對齊紙張後重啟",
}

@mcp.tool()
def check_inventory(product: str) -> dict:
    """查詢 HP 產品庫存數量與位置"""
    return _INVENTORY.get(product, {"stock": "未知", "location": "查無此產品"})

@mcp.tool()
def check_warranty(serial_number: str) -> dict:
    """根據序號確認 HP 產品保固狀態"""
    return _WARRANTY.get(serial_number, {"status": "查無資料", "expire": None})

@mcp.tool()
def get_repair_sop(issue: str) -> str:
    """取得 HP 常見問題的維修 SOP"""
    for key, sop in _SOP.items():
        if key in issue:
            return sop
    return "請聯絡 HP 技術支援：0800-012-345"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("hp_mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ hp_mcp_server.py 已建立")

---
## 2. MCP Client 串接 LangGraph

`langchain-mcp-adapters` 提供 `MultiServerMCPClient`，讓 LangGraph Agent 自動發現並載入 MCP Server 上的工具。

```
LangGraph Agent
     │
     └─ MultiServerMCPClient
          └─ stdio → hp_mcp_server.py
                       ├─ check_inventory
                       ├─ check_warranty
                       └─ get_repair_sop
```

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 設定 MCP Server 連線（stdio 模式）
mcp_config = {
    "hp-tools": {
        "command": "python",
        "args": ["hp_mcp_server.py"],
        "transport": "stdio",
    }
}

async def run_mcp_agent(query: str):
    async with MultiServerMCPClient(mcp_config) as client:
        # 從 MCP Server 自動載入工具
        tools = client.get_tools()
        print(f"📦 從 MCP Server 載入 {len(tools)} 個工具：{[t.name for t in tools]}")

        # 建立 ReAct Agent
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": [("user", query)]})

        # 印出回應
        for msg in result["messages"]:
            role = msg.__class__.__name__
            if hasattr(msg, 'content') and msg.content:
                print(f"  [{role}] {msg.content[:300]}")
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  [Tool Call] {tc['name']}({tc['args']})")

# 測試 1：查詢庫存
print("=== 測試 1：查詢庫存 ===")
await run_mcp_agent("HP EliteBook 840 還有庫存嗎？在哪個倉庫？")

In [ ]:
# 測試 2：確認保固
print("=== 測試 2：確認保固 ===")
await run_mcp_agent("我的序號是 SN12345，請確認保固是否還有效？")

In [ ]:
# 測試 3：複合查詢
print("=== 測試 3：複合查詢 ===")
await run_mcp_agent("我的序號 SN99999 印表機卡紙了，保固還有效嗎？該怎麼修？")

---
## 3. MCP Resources — 靜態知識的標準存取方式

除了 `@mcp.tool()` 呼叫動態工具，MCP 還支援 `@mcp.resource()` 提供靜態資料（如產品目錄）。

| 類型 | 用途 | 觸發方式 |
|------|------|----------|
| `tool` | 動態操作（查詢、計算） | Agent 主動呼叫 |
| `resource` | 靜態資料（目錄、規格） | URI 存取 `kb://products` |
| `prompt` | 預設 Prompt 範本 | Client 請求 |

In [ ]:
# 示範 Resource 的寫法（不需執行 Server）
resource_example = '''
from fastmcp import FastMCP

mcp = FastMCP("HP 知識庫")

PRODUCT_CATALOG = [
    {"line": "EliteBook", "segment": "商務筆電", "range": "800 / 1000 系列"},
    {"line": "ZBook",     "segment": "工作站筆電", "range": "Studio / Fury"},
    {"line": "LaserJet",  "segment": "雷射印表機", "range": "Pro / Enterprise"},
]

# Resource：靜態產品目錄
@mcp.resource("kb://products")
def list_products() -> list:
    """列出 HP 知識庫中的產品線"""
    return PRODUCT_CATALOG

# Tool：動態向量搜尋
@mcp.tool()
def search_kb(query: str, top_k: int = 5) -> list[str]:
    """搜尋 HP 產品知識庫"""
    # vectorstore.similarity_search(query, k=top_k)
    return [f"[模擬結果] 與 '{query}' 相關的文件內容..."] * top_k
'''

print(resource_example)
print("💡 重點：同一個 MCP Server 可同時服務多個 Agent（Agentic RAG + Deep Research）")

---
## 4. 企業部署架構

HP 場景下的 MCP 部署方式：

```
┌─────────────────────────────────────────┐
│              LangGraph Agents           │
│  ┌──────────┐  ┌──────────┐  ┌────────┐ │
│  │ RAG Agent│  │Voice Agt │  │Research│ │
│  └────┬─────┘  └────┬─────┘  └───┬────┘ │
└───────┼─────────────┼─────────────┼──────┘
        │  MCP Client │             │
┌───────▼─────────────▼─────────────▼──────┐
│              MCP Servers                  │
│  ┌──────────┐  ┌──────────┐  ┌─────────┐ │
│  │庫存 Server│  │保固 Server│  │SOP Server│ │
│  └──────────┘  └──────────┘  └─────────┘ │
└───────────────────────────────────────────┘
```

**優點**：工具更新只需改 Server，所有 Agent 自動獲得最新版本，不需重新部署。

In [ ]:
# claude_desktop_config.json 的設定方式（供參考）
import json

claude_config = {
    "mcpServers": {
        "hp-inventory": {
            "command": "python",
            "args": ["hp_mcp_server.py"]
        },
        "hp-knowledge": {
            "command": "python",
            "args": ["hp_kb_server.py"]
        }
    }
}

print("claude_desktop_config.json:")
print(json.dumps(claude_config, indent=2, ensure_ascii=False))
print("\n✅ 將此設定放入 ~/Library/Application Support/Claude/claude_desktop_config.json")
print("   Claude Desktop 重啟後即可看到 HP Tools")